In [14]:
import sys, os
sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath(".")), "Code", 'tools'))
from pathlib import Path

# Change root directory to the repo root (Jupyter: __file__ is not defined)
def find_repo_root(start=Path.cwd()):
	for p in [start] + list(start.parents):
		if (p / 'Code').exists() or (p / '.git').exists() or (p / 'local_repo').exists():
			return p
	return start

repo_root = find_repo_root()
data_root = repo_root.parent.parent / 'Data' / 'LIS' / 'code'
json_root = repo_root.parent.parent / 'Data' / 'CT json'

os.chdir(repo_root)
sys.path.append(str(repo_root / 'Code' / 'tools'))

from lis_automizer import run_lis_job

In [15]:
# Open json file
json_path = json_root / 'LIS_new_voiv' / 'LIS_2005.json'

with open(json_path, 'r') as f:
    json = f.read()


In [16]:
body = """# Prepare data
library(glue)
library(dplyr)
library(magrittr)
library(purrr)
library(ggplot2)
library(tidyr)
library(labelled)
library(jsonlite)
all_lissyrtools_scripts <- fs::dir_ls("/media/user/lissyrtools/")
invisible(purrr::map(all_lissyrtools_scripts, ~ source(.x)))
databases_poland_p <- read_lissy_files(files = c("pl05p"), full_year_name = TRUE)
databases_poland_h <- read_lissy_files(files = c("pl05h"), full_year_name = TRUE)
# ---------------------------------------------------------------
# HELPER: decode region labels
# as.factor() works on pl05h region_c (confirmed by diagnostics)
# Strip "[XX]" prefix, lowercase, fix zachodnio-pomorskie
# ---------------------------------------------------------------
clean_region_label <- function(x) {
  r <- as.character(as.factor(x))
  r <- gsub("^\\\[[0-9]+\\\]", "", r)
  r <- tolower(r)
  r <- gsub("zachodnio-pomorskie", "zachodniopomorskie", r)
  r
}
# Decode region_c on df_h BEFORE joining (attributes intact)
databases_poland <- purrr::map2(databases_poland_p, databases_poland_h, function(df_p, df_h) {
  region_lookup <- df_h %>%
    mutate(region_c = clean_region_label(region_c)) %>%
    select(hid, region_c, locsz_c)
  df_p %>%
    left_join(region_lookup, by = "hid")
})
databases_poland <- purrr::map(databases_poland, function(df) {
  df %>% mutate(
    pitotalnet = pitotal - pxitsc,
    picash     = pitotal - pi13
  )
})
# ---------------------------------------------------------------
# RIM WEIGHTING DATA
# PASTE YOUR FULL JSON BETWEEN THE SINGLE QUOTES BELOW
# ---------------------------------------------------------------""" + f"""
age_sex_targets <- fromJSON('{json}', simplifyVector = FALSE) """ + """# ---------------------------------------------------------------
# HELPER FUNCTIONS
# NOTE: sex is a factor in pl05h/pl05p — must use as.numeric(sex)
# ---------------------------------------------------------------
get_age_bracket <- function(age) {
  case_when(
    age <=  4 ~ "0-4",
    age <=  9 ~ "5-9",
    age <= 14 ~ "10-14",
    age <= 19 ~ "15-19",
    age <= 24 ~ "20-24",
    age <= 29 ~ "25-29",
    age <= 34 ~ "30-34",
    age <= 39 ~ "35-39",
    age <= 44 ~ "40-44",
    age <= 49 ~ "45-49",
    age <= 54 ~ "50-54",
    age <= 59 ~ "55-59",
    age <= 64 ~ "60-64",
    age <= 69 ~ "65-69",
    TRUE      ~ "70 i wi\u0119cej"
  )
}
get_sex_label <- function(sex) {
  case_when(
    as.numeric(sex) == 1 ~ "m\u0119\u017czy\u017ani",
    as.numeric(sex) == 2 ~ "kobiety",
    TRUE                 ~ NA_character_
  )
}
# ---------------------------------------------------------------
# ASSIGN GROUPS AND COMPUTE RIM WEIGHTS
# ---------------------------------------------------------------
df_2005 <- databases_poland[[1]] %>%
  filter(region_c %in% names(age_sex_targets)) %>%
  mutate(
    age_bracket = get_age_bracket(age),
    sex_label   = get_sex_label(sex)
  ) %>%
  filter(!is.na(age_bracket), !is.na(sex_label)) %>%
  group_by(region_c, age_bracket, sex_label) %>%
  mutate(
    n_in_cell     = n(),
    census_target = as.numeric(mapply(function(reg, ab, sl) {
      cell_key <- paste(ab, "\u00d7", sl)
      target   <- age_sex_targets[[reg]][["2005"]][["E_age_sex_2000"]][[cell_key]]
      if (is.null(target)) NA_real_ else as.numeric(target)
    }, region_c, age_bracket, sex_label)),
    rim_weight = census_target / n_in_cell
  ) %>%
  ungroup() %>%
  filter(!is.na(rim_weight))
# ---------------------------------------------------------------
# WEIGHTED REGIONAL STATISTICS
# ---------------------------------------------------------------
calculate_regional_stats <- function(df, region_value) {
  df_clean <- df %>% filter(region_c == region_value)
  if (nrow(df_clean) == 0) return(NULL)
  overall_means <- df_clean %>%
    summarise(
      mean_pitotalnet = sum(pitotalnet * rim_weight, na.rm = TRUE) / sum(rim_weight[!is.na(pitotalnet)]),
      mean_picash     = sum(picash     * rim_weight, na.rm = TRUE) / sum(rim_weight[!is.na(picash)]),
      year            = first(year)
    )
  calculate_groups <- function(income_var_name) {
    valid_data <- df_clean %>%
      filter(!is.na(.data[[income_var_name]])) %>%
      arrange(.data[[income_var_name]]) %>%
      mutate(
        cum_weight   = cumsum(rim_weight),
        total_weight = sum(rim_weight),
        percentile   = cum_weight / total_weight,
        group = case_when(
          percentile <= 0.50 ~ "Bottom_50",
          percentile <= 0.90 ~ "P50_90",
          percentile >  0.90 ~ "Top_10",
          TRUE               ~ NA_character_
        )
      )
    group_means <- valid_data %>%
      group_by(group) %>%
      summarise(
        mean_value = sum(.data[[income_var_name]] * rim_weight) / sum(rim_weight),
        .groups = "drop"
      ) %>%
      pivot_wider(
        names_from   = group,
        values_from  = mean_value,
        names_prefix = paste0(income_var_name, "_")
      )
    top_1_mean <- valid_data %>%
      filter(percentile > 0.99) %>%
      summarise(mean_value = sum(.data[[income_var_name]] * rim_weight) / sum(rim_weight)) %>%
      pull(mean_value)
    group_means[[paste0(income_var_name, "_Top_1")]] <- top_1_mean
    return(group_means)
  }
  pitotalnet_groups <- calculate_groups("pitotalnet")
  picash_groups     <- calculate_groups("picash")
  bind_cols(overall_means, pitotalnet_groups, picash_groups)
}
# ---------------------------------------------------------------
# OUTPUT FORMATTING
# ---------------------------------------------------------------
print_region_dict <- function(year_value, region_value, region_data) {
  if (is.null(region_data) || nrow(region_data) == 0) {
    cat(sprintf("\n# No data for year %d, region %s\n", year_value, region_value))
    return()
  }
  region_name <- gsub("[^A-Za-z0-9]", "_", as.character(region_value))
  cat(sprintf("\n# Year: %d, Region: %s\n", year_value, region_value))
  cat(sprintf("data_region_%d_%s = {\n", year_value, region_name))
  cat(sprintf("    'year': [%d],\n",                    region_data$year))
  cat(sprintf("    'pitotalnet': [%.2f],\n",             region_data$mean_pitotalnet))
  cat(sprintf("    'picash': [%.2f],\n",                 region_data$mean_picash))
  cat(sprintf("    'pitotalnet_Bottom_50': [%.2f],\n",   region_data$pitotalnet_Bottom_50))
  cat(sprintf("    'pitotalnet_P50_90': [%.2f],\n",      region_data$pitotalnet_P50_90))
  cat(sprintf("    'pitotalnet_Top_10': [%.2f],\n",      region_data$pitotalnet_Top_10))
  cat(sprintf("    'pitotalnet_Top_1': [%.2f],\n",       region_data$pitotalnet_Top_1))
  cat(sprintf("    'picash_Bottom_50': [%.2f],\n",       region_data$picash_Bottom_50))
  cat(sprintf("    'picash_P50_90': [%.2f],\n",          region_data$picash_P50_90))
  cat(sprintf("    'picash_Top_10': [%.2f],\n",          region_data$picash_Top_10))
  cat(sprintf("    'picash_Top_1': [%.2f]\n",            region_data$picash_Top_1))
  cat("}\n")
}
# ---------------------------------------------------------------
# MAIN LOOP
# ---------------------------------------------------------------
cat("\n========== REGIONAL DATA DICTIONARIES (2005, RIM WEIGHTED BY REGION) ==========\n")
all_regions <- df_2005 %>%
  select(region_c) %>%
  distinct() %>%
  pull(region_c) %>%
  sort()
year_value <- first(df_2005$year)
for (region in all_regions) {
  region_stats <- calculate_regional_stats(df_2005, region)
  print_region_dict(year_value, region, region_stats)
}"""

In [17]:
# Save body as .txt file
out_path = data_root / 'body.txt'

# Dump body to .txt file
with open(out_path, 'w') as f:
    f.write(body)

In [8]:
run_lis_job(
    code = body,
    job_title="Test job from LISAutomizer",
    project="LIS",
    package="R",
    wait_for_result=False,
    max_wait=300,
)

## Multi-job session (keeps one browser open)

In [ ]:
# Multi-job session: reuse one browser for multiple submissions
jobs = [
    {"code": 'print("hello from job 1")', "title": "batch job 1"},
    {"code": 'print("hello from job 2")', "title": "batch job 2"},
]

with LISAutomizer() as lis:
    for job in jobs:
        result = lis.submit_job(
            code=job["code"],
            job_title=job["title"],
            project="LIS",
            package="R",
            wait_for_result=False,  # fire-and-forget mode
        )
        print(f"Submitted: {job['title']}")

# LIS LISSY Automizer

Automate job submission to the [LIS Data Center LISSY](https://webui.lisdatacenter.org) web interface via Selenium + Safari.

**Setup:**
```bash
pip install selenium
```
Safari WebDriver is built into macOS. Enable it once via:
```bash
safaridriver --enable
```

**Credentials** are handled automatically:
- User ID: `jslowi` (hardcoded)
- Password: read from `password.rtf` (located two directories above the repo root)